In [ ]:
!pip install transformers datasets peft sentence-transformers faiss-cpu accelerate -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 79.4 MB/s eta 0:00:00


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
import torch, os


In [ ]:
# =====================================
# Step 3: Load Dataset
# =====================================
# Your dataset should be in JSON format with keys: "prompt" and "completion"
dataset = load_dataset("json", data_files='clean_healthcare_chatbot_15k_expanded_codemixed.jsonl')['train']
print("Sample example:")
print(dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

Sample example:
{'prompt': 'Naku stomach pain undi, em cheyyali?', 'completion': 'Stomach pain vunte first light food teesukondi and heavy/spicy/oily food avoid cheyyandi. Warm water or heat pad apply chesi abdominal muscles relax cheyyandi. If vomiting, blood in vomit or severe continuous pain untunte immediate hospital visit avasaram. Mild indigestion ki antacid doctor advice tho teesukovachu but self-medication avoid cheyyandi. Keep hydration and small frequent meals preserve cheyyandi; symptom diary maintain cheyyandi.'}


In [ ]:
# =====================================
# Step 4: Load GPT-2 Small and Apply LoRA (Supports Resuming)
# =====================================
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
print("🚀 Starting new training...")
model = GPT2LMHeadModel.from_pretrained(model_name)
lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        target_modules=["c_attn", "c_proj"],
        lora_dropout=0.1,
        bias="none"
)
model = get_peft_model(model, lora_config)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

🚀 Starting new training...


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [ ]:
def tokenize(examples):
    # examples is a dict of lists when batched=True
    texts = [p + c for p, c in zip(examples["prompt"], examples["completion"])]
    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=128)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize, batched=True)


Map:   0%|          | 0/10500 [00:00<?, ? examples/s]

In [ ]:
!pip install -U transformers

In [ ]:
training_args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=2,  # T4 friendly
    gradient_accumulation_steps=8,  # effective batch size 16
    learning_rate=2e-4,
    num_train_epochs=30,
    logging_steps=50,
    save_strategy="epoch",
    fp16=True,  # memory-efficient optimizer
    report_to="none"
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

In [ ]:
trainer.train()
model.save_pretrained("./output")
tokenizer.save_pretrained("./output")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,4.512300
100,2.935300
150,2.226900
200,1.742800
250,1.384900
300,1.187900
350,0.960200
400,0.739400
450,0.606700
500,0.479300


Step,Training Loss
50,4.512300
100,2.935300
150,2.226900
200,1.742800
250,1.384900
300,1.187900
350,0.960200
400,0.739400
450,0.606700
500,0.479300


('./output/tokenizer_config.json',
 './output/special_tokens_map.json',
 './output/vocab.json',
 './output/merges.txt',
 './output/added_tokens.json')

In [ ]:
import shutil

shutil.make_archive("output30epochs", 'zip', "output")



'/content/output30epochs.zip'

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json

# 1. Load text KB
with open("kb_docs_rich.txt", "r", encoding="utf-8") as f:
    kb_texts = [line.strip() for line in f if line.strip()]

# 2. Encode using multilingual model
embed_model = SentenceTransformer("l3cube-pune/indic-sentence-bert-nli")
kb_embeddings = embed_model.encode(kb_texts, convert_to_numpy=True, show_progress_bar=True)

# 3. Build FAISS index
dim = kb_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(kb_embeddings)

# 4. Save everything
faiss.write_index(index, "faiss_index.index")
with open("kb_texts.json", "w", encoding="utf-8") as f:
    json.dump(kb_texts, f, ensure_ascii=False, indent=2)

print(f"✅ FAISS index built with {len(kb_texts)} entries.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ FAISS index built with 60 entries.


In [ ]:
import torch, json, faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# 1️⃣ Correct path to your model folder
model_dir = "/content/output"

# 2️⃣ Load model + tokenizer from Drive
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype=torch.float16)

# 3️⃣ Create text generation pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")

# 4️⃣ Load FAISS index and KB
index = faiss.read_index("/content/faiss_index.index")
with open("/content/kb_texts.json", encoding="utf-8") as f:
    kb_texts = json.load(f)

# 5️⃣ Load sentence embedding model
embed_model = SentenceTransformer("l3cube-pune/indic-sentence-bert-nli")

# 6️⃣ Define retrieval function
def retrieve_top_k(query, k=3):
    query_emb = embed_model.encode([query])
    D, I = index.search(query_emb, k)
    return [kb_texts[i] for i in I[0]]

# 7️⃣ Define RAG response function
def rag_response(user_query):
    context = retrieve_top_k(user_query, k=3)
    prompt = f"### Instruction:\nUser: {user_query}\nContext: {' '.join(context)}\n### Response:\n"
    response = generator(prompt, max_new_tokens=100, temperature=0.7, top_p=0.9)[0]["generated_text"]
    return response.split("### Response:")[-1].strip()


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


In [ ]:
import json, random

all_data = []
with open("/content/clean_healthcare_chatbot_15k_expanded_codemixed.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:  # skip empty lines
            try:
                all_data.append(json.loads(line))
            except json.JSONDecodeError:
                print("Skipping invalid line:", line)

# =====================================
# 3️⃣ Split into train/test
# =====================================
random.seed(42)
random.shuffle(all_data)
test_size = 500  # you can choose 500-1000 for evaluation
test_data = all_data[:test_size]
train_data = all_data[test_size:]

In [ ]:
!pip install evaluate bert_score sacrebleu rouge_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.0 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=3c12fb53ebbbc0e198ab1a0fd4137bcf722579fbbb6fdcc677dbc30fae107f03
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
from evaluate import load
bertscore = load("bertscore")

generated_texts = []
reference_texts = []
for entry in test_data:
    user_query = entry["prompt"].split("### Instruction:\n")[-1].split("\n\n### Context:")[0]
    ref = entry["completion"]
    gen = rag_response(user_query)
    generated_texts.append(gen)
    reference_texts.append(ref)

bertscore_results = bertscore.compute(predictions=generated_texts, references=reference_texts, lang="en")
print("BERTScore F1:", sum(bertscore_results["f1"])/len(bertscore_results["f1"]))

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore F1: 0.8344425730705262


In [ ]:
from transformers import AutoModelForCausalLM
from tqdm import tqdm # Import tqdm here

model.eval()
def compute_ppl(sentences):
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for sent in tqdm(sentences):
            inputs = tokenizer(sent, return_tensors="pt", truncation=True, max_length=128)
            labels = inputs["input_ids"]
            inputs = {k: v.to(model.device) for k, v in inputs.items()} # Move inputs to model device
            outputs = model(**inputs, labels=labels)
            total_loss += outputs.loss.item() * labels.size(1)
            total_tokens += labels.size(1)
    import math
    ppl = math.exp(total_loss / total_tokens)
    return ppl

ppl = compute_ppl(generated_texts)
print("Perplexity on generated responses:", ppl)

100%|██████████| 500/500 [00:20<00:00, 24.93it/s]

Perplexity on generated responses: 119.58883347363532


In [ ]:
from evaluate import load
import numpy as np

# Load metrics
bleu = load("sacrebleu")
rouge = load("rouge")

# Collect generations and references
generated_texts = []
reference_texts = []

for entry in test_data:
    user_query = entry["prompt"].split("### Instruction:\n")[-1].split("\n\n### Context:")[0]
    ref = entry["completion"]
    gen = rag_response(user_query)
    generated_texts.append(gen)
    reference_texts.append(ref)

# -------------------------------
# 1️⃣ BLEU
# -------------------------------
bleu_results = bleu.compute(predictions=generated_texts, references=[[r] for r in reference_texts])
bleu_score = bleu_results["score"]

# -------------------------------
# 2️⃣ ROUGE-L
# -------------------------------
rouge_results = rouge.compute(predictions=generated_texts, references=reference_texts)
rouge_l = rouge_results["rougeL"]

# -------------------------------
# 📊 Print Summary
# -------------------------------
print("\n===== Evaluation Summary =====")
print(f"BLEU: {bleu_score:.2f}")
print(f"ROUGE-L: {rouge_l:.3f}")

# Optionally save to file
with open("bleu_rouge_results.txt", "w") as f:
    f.write(f"BLEU: {bleu_score:.2f}\n")
    f.write(f"ROUGE-L: {rouge_l:.3f}\n")


===== Evaluation Summary =====
BLEU: 5.23
ROUGE-L: 0.164


In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

# Load a multilingual Sentence-BERT model
model = SentenceTransformer('l3cube-pune/indic-sentence-bert-nli')

generated_texts = []
reference_texts = []

# Example: assuming your test_data is already loaded
for entry in test_data:
    user_query = entry["prompt"].split("### Instruction:\n")[-1].split("\n\n### Context:")[0]
    ref = entry["completion"]
    gen = rag_response(user_query)
    generated_texts.append(gen)
    reference_texts.append(ref)

# Compute embeddings
gen_embeds = model.encode(generated_texts, convert_to_tensor=True)
ref_embeds = model.encode(reference_texts, convert_to_tensor=True)

# Compute cosine similarity
cosine_scores = util.cos_sim(gen_embeds, ref_embeds)

# Take diagonal (pairwise scores between matching responses)
pairwise_scores = cosine_scores.diag().cpu().numpy()

# Average similarity
mean_similarity = np.mean(pairwise_scores)
print(f"Average Semantic Similarity: {mean_similarity:.4f}")

Average Semantic Similarity: 0.6092


In [ ]:
queries = [
    "Naku headache vachindi, em cheyyali?",
    "Cough valla em chesthavu?",
    "Throat pain ki emi cheyyali?",
    "Cold vachindi, em cheyyali?"
]

for q in queries:
    print(f"\n🧍 User: {q}")
    codemixed = rag_response(q)  # your fine-tuned GPT-2 + RAG outpu
    print(f"🤖 Bot (Code-mixed Telugu): {codemixed}")


🧍 User: Naku headache vachindi, em cheyyali?
🤖 Bot (Code-mixed Telugu): Maintain healthy weight, hydration and rest; examination and tests doctor advise chesaru.

🧍 User: Cough valla em chesthavu?
🤖 Bot (Code-mixed Telugu): Store cold foods and fluids teesukondi; spicy and cold items avoid cheyyandi. If cough continues beyond 7–10 rojulu, medical evaluation recommended under specialist guidance.

🧍 User: Throat pain ki emi cheyyali?
🤖 Bot (Code-mixed Telugu): Rest, gentle stretching and warm bath chala help chestayi. Avoid heavy lifting and sudden movements; maintain healthy weight to reduce headache. If pain continues beyond 7–10 rojulu, medical evaluation recommended under specialist guidance.

🧍 User: Cold vachindi, em cheyyali?
🤖 Bot (Code-mixed Telugu): Sudden confusion, disorientation or rapid decline unte doctor ni immediate ga kalavandi; neurological evaluation important. Medications and cognitive therapies sometimes recommended under specialist guidance.


In [ ]:
import gradio as gr
import faiss, json, numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# ==========================================
# 🔧 Initialization
# ==========================================
print("🔁 Initializing models...")

# embedding model
embed_model = SentenceTransformer("l3cube-pune/indic-sentence-bert-nli", device="cuda" if torch.cuda.is_available() else "cpu")

# FAISS + KB
index = faiss.read_index("faiss_index.index")
with open("kb_texts.json", "r", encoding="utf-8") as f:
    kb_texts = json.load(f)

# RAG generation model
generator = pipeline(
    "text-generation",
    model="./output",
    tokenizer="./output",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Translation model
TRANSLATOR_MODEL = "aryaumesh/english-to-telugu"
translator_tokenizer = AutoTokenizer.from_pretrained(TRANSLATOR_MODEL)
translator_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATOR_MODEL)

SRC_LANG = "en_XX"
TGT_LANG = "te_IN"
translator_tokenizer.src_lang = SRC_LANG

print("🔥 All models loaded successfully.")


# ==========================================
# 🧠 Retrieval + RAG Generation
# ==========================================
def retrieve_top_k(query, k=3):
    q_emb = embed_model.encode([query]).astype("float32")
    D, I = index.search(q_emb, k)

    # remove -1 and duplicates
    unique_contexts = []
    for i in I[0]:
        if i != -1 and kb_texts[i] not in unique_contexts:
            unique_contexts.append(kb_texts[i])

    return unique_contexts


def translate_to_telugu(text):
    """Translate code-mixed English Telugu → pure Telugu"""
    try:
        encoded = translator_tokenizer(text, return_tensors="pt")
        generated_tokens = translator_model.generate(
            **encoded,
            forced_bos_token_id=translator_tokenizer.lang_code_to_id[TGT_LANG],
            max_length=150,
            num_beams=5,
            early_stopping=True,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2
        )
        return translator_tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
    except:
        return "⚠️ Translation Failed."


def rag_response(user_query, k=3):
    context_list = retrieve_top_k(user_query)
    context = "\n".join(context_list)

    prompt = f"""
### Instruction:
Help the user with a friendly medical suggestion.

### User Query:
{user_query}

### Knowledge:
{context}

### Response:
"""

    result = generator(prompt, max_new_tokens=250, temperature=0.7, top_p=0.9)[0]["generated_text"]
    final_resp = result.replace(prompt, "").strip()

    return final_resp, context_list


# ==========================================
# 💬 Chat Flow
# ==========================================
def chat_function(message, history):
    code_mixed, context_used = rag_response(message)

    # Translate model response
    telugu_translation = translate_to_telugu(code_mixed)

    # Translate retrieved medical reference
    translated_contexts = [translate_to_telugu(c) for c in context_used]

    response = f"""
🗣 **Code-Mixed Response:**
{code_mixed}

🇮🇳 **Pure Telugu Translation:**
{telugu_translation}

---

📌 **Retrieved Medical Reference (English):**
"""

    for c in context_used:
        response += f"• {c[:180]}...\n"

    response += "\n\n📖 **Retrieved Reference (Telugu Translation):**\n"

    for t in translated_contexts:
        response += f"• {t[:180]}...\n"

    return response



# ==========================================
# 🎨 UI Styling
# ==========================================
custom_css = """
#chatbot {height: 600px !important;}
.gradio-container {background-color: #0d1117 !important;}
.message.user {background-color: #1f6feb !important;color: white;border-radius: 14px;padding: 10px;}
.message.bot {background-color: #161b22 !important;color: #e6edf3 !important;border: 1px solid #30363d;border-radius: 14px;padding: 10px;}
textarea {background-color: #161b22 !important;border-radius: 10px;color: white;border:1px solid #30363d;}
button {background:#238636;color:white;border-radius:8px;font-weight:bold;}
button:hover {background:#2ea043;}
"""


# ==========================================
# 🚀 Launch App
# ==========================================
chatbot = gr.ChatInterface(
    fn=chat_function,
    title="🏥 Telugu HealthGPT",
    description="Ask your health question in Telugu, English or mix.\nExample: `Naku headache undi. Em cheyyali?`",
    css=custom_css,
    examples=[
        ["Naku headache undi, em cheyyali?"],
        ["Cold and cough ki home remedies cheppu."],
        ["Fever vachindi 2 days ninchi. Emi cheyyali?"],
    ]
)

chatbot.queue().launch(share=True)


🔁 Initializing models...


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

🔥 All models loaded successfully.


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c3fd998dd2a2749496.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
